# Project 14: Traffic Congestion Prediction (Duplicate/Validation of Project 05)
**Team No.:** 24  
**Team Members:** Smruti Rekha Panda; Simran Sahu; Prativa Panda; Rajashree Samal  
**Proposed Hybrid Model:** Diffusion GCN + Temporal Transformer  
**Dataset:** PEMS03 / PEMS04 / PEMS07 / PEMS08 traffic datasets  
**Source:** https://www.kaggle.com/datasets/elmahy/pems-dataset


## 0. Setup — Environment, Imports, Reproducibility


In [1]:
!pip -q install kaggle tqdm tabulate
import os, json, random, glob, subprocess, sys, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print("Using device:", DEVICE)


Using device: cuda:0


### CONFIG


In [2]:
CONFIG = {
 "project_no":"14", "project_name":"Traffic Congestion Prediction (Duplicate/Validation of Project 05)", "team_no":"24",
 "task_type":"regression", "kaggle_dataset_slug":"elmahy/pems-dataset",
 "target_column":None, "id_columns":[], "time_column":None,
 "split_ratios":{"train":0.70,"val":0.15,"test":0.15}, "random_seed":SEED,
 "data_raw_dir":"data/14/raw", "data_processed_dir":"data/14/processed", "figures_dir":"data/14/figures", "results_dir":"data/14/results",
 "checkpoints_dir":"data/14/results/checkpoints",
 "reports_dir":"data/14/reports", "batch_size":64, "epochs":30, "patience":5
}
# Create every directory any later cell writes to (top-level dirs AND the nested
# checkpoints/ subdirectory) - a loop that only makes top-level dirs is exactly the
# bug that caused "Parent directory results does not exist" on torch.save().
for key in ["data_raw_dir","data_processed_dir","figures_dir","results_dir","checkpoints_dir","reports_dir"]: os.makedirs(CONFIG[key],exist_ok=True)
CONFIG

{'project_no': '14',
 'project_name': 'Traffic Congestion Prediction (Duplicate/Validation of Project 05)',
 'team_no': '24',
 'task_type': 'regression',
 'kaggle_dataset_slug': 'elmahy/pems-dataset',
 'target_column': None,
 'id_columns': [],
 'time_column': None,
 'split_ratios': {'train': 0.7, 'val': 0.15, 'test': 0.15},
 'random_seed': 42,
 'data_raw_dir': 'data/14/raw',
 'data_processed_dir': 'data/14/processed',
 'figures_dir': 'data/14/figures',
 'results_dir': 'data/14/results',
 'checkpoints_dir': 'data/14/results/checkpoints',
 'reports_dir': 'data/14/reports',
 'batch_size': 64,
 'epochs': 30,
 'patience': 5}

## 1. Dataset Download


In [3]:
raw=Path(CONFIG["data_raw_dir"])
if not any(raw.rglob("*")):
    subprocess.run(["kaggle","datasets","download","-d",CONFIG["kaggle_dataset_slug"],"-p",str(raw),"--unzip"],check=True)
for z in raw.rglob("*.zip"):
    import zipfile
    with zipfile.ZipFile(z) as f: f.extractall(z.parent/z.stem)
raw_files=[p for p in raw.rglob("*") if p.is_file()]
assert raw_files, "Dataset download produced no files. Configure Kaggle credentials in Colab and rerun."
assert sum(p.stat().st_size for p in raw_files)>1024, "Downloaded content is unexpectedly small."
print(f"Discovered {len(raw_files)} files; {sum(p.stat().st_size for p in raw_files)/2**20:.1f} MiB")


Discovered 9 files; 105.9 MiB


## 2. Load Raw Data


In [4]:
# Recursively search every subfolder (Kaggle downloads often nest the real file a level or
# two deep) and accept the common array formats PEMS-style downloads show up in: .npz/.npy,
# HDF5 (.h5/.hdf5, commonly under an "df"/"data" key), and CSV files that hold a speed/flow
# matrix (rows=timesteps, cols=sensors).
files=list(Path(CONFIG["data_raw_dir"]).rglob("*")); arrays=[]; seen=[]
for f in files:
    if not f.is_file(): continue
    suf=f.suffix.lower()
    try:
        if suf==".npz":
            z=np.load(f); arrays.extend([z[k] for k in z.files if np.asarray(z[k]).ndim>=2]); seen.append(f)
        elif suf==".npy":
            arrays.append(np.load(f)); seen.append(f)
        elif suf in (".h5",".hdf5"):
            import h5py
            with h5py.File(f,"r") as hf:
                for k in hf.keys():
                    arr=np.asarray(hf[k])
                    if arr.ndim>=2: arrays.append(arr)
            seen.append(f)
        elif suf==".csv":
            arr=pd.read_csv(f).select_dtypes(include=[np.number]).to_numpy()
            if arr.ndim>=2 and arr.size>0: arrays.append(arr)
            seen.append(f)
    except Exception as e: print("Skipped",f,e)
assert arrays,f"No readable PEMS NumPy array found. Files seen under {CONFIG['data_raw_dir']}: {[str(p) for p in files][:50]}"
series=max(arrays,key=lambda a:a.size); series=np.asarray(series)
if series.ndim==3: series=series[...,0]
while series.ndim>2: series=series.reshape(series.shape[0],-1)
if series.shape[0]<series.shape[1]: series=series.T
assert series.shape[0]>100 and series.shape[1]>1
print("traffic matrix",series.shape)


traffic matrix (28224, 883)


## 3. Exploratory Data Analysis (EDA) + Data Quality Memo


In [5]:
missing=float(np.isnan(series).mean())
memo = "\n".join([
    "# Data quality memo",
    f"- Matrix shape: {series.shape}.",
    f"- Missing fraction: {missing:.4f}.",
    "- Chronological splitting prevents future leakage.",
    "- Scaling is fit on the training interval only.",
    "",
    "## Known limitations",
    "- Adjacency is a correlation-threshold graph built from the training split, not the",
    "  physical PEMS road-sensor topology, so diffusion hops approximate true propagation.",
    "- Only one of PEMS03/04/07/08 is used per run (whichever array file dominates the",
    "  Kaggle download); results are not pooled across all four datasets.",
    "- Lookback/horizon are fixed constants, matching Project 05's setup for a like-for-like",
    "  validation rather than per-dataset tuning.",
])
Path(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md")).write_text(memo,encoding="utf-8")

652

## 4. Preprocessing & Feature Engineering


Feature construction is performed after splitting; every learned imputer, scaler, encoder, graph, and vocabulary is fit on training data only.


## 5. Train / Validation / Test Split


In [6]:
n=len(series); ntr=int(.70*n); nv=int(.15*n)
raw_series=np.asarray(series,dtype="float32").copy()
# Causal preprocessing: training medians are the only fitted fill values; forward fill uses past observations only.
train_median=np.nanmedian(raw_series[:ntr],axis=0)
causal=pd.DataFrame(raw_series).ffill().to_numpy(dtype="float32")
causal=np.where(np.isnan(causal),train_median,causal).astype("float32")
assert np.isfinite(causal).all()
scaler=StandardScaler().fit(causal[:ntr]); scaled=scaler.transform(causal).astype("float32"); LOOKBACK=24; HORIZON=1
def windows(a,start,end):
    xs=[]; ys=[]
    for t in range(max(start,LOOKBACK),end-HORIZON+1): xs.append(a[t-LOOKBACK:t]); ys.append(a[t:t+HORIZON].squeeze(0))
    return np.stack(xs),np.stack(ys)
Xtr,ytr=windows(scaled,0,ntr); Xv,yv=windows(scaled,ntr,ntr+nv); Xte,yte=windows(scaled,ntr+nv,n)
assert Xtr.shape[0] and Xv.shape[0] and Xte.shape[0]
assert ntr < ntr+nv < n
Path(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json")).write_text(json.dumps({"train":len(Xtr),"val":len(Xv),"test":len(Xte),"lookback":LOOKBACK,"imputation":"training-median plus causal forward-fill"},indent=2),encoding="utf-8")


129

In [7]:
# --- Target sanity check (catch a degenerate/near-constant target early, e.g. from a
# row-limited/sorted read that grabs a degenerate block of the series) ---
_tgt = ytr.reshape(-1)
print(f"Target (ytr) stats: min={_tgt.min():.4f} max={_tgt.max():.4f} mean={_tgt.mean():.4f} std={_tgt.std():.4f}")
assert _tgt.std() > 1e-6, f"DEGENERATE TARGET: near-zero variance ({_tgt.std()}) - check upstream filtering."
if len(_tgt) < 100:
    print(f"WARNING: very small target sample size ({len(_tgt)} rows) - check upstream row-limiting logic.")

Target (ytr) stats: min=-4.8356 max=24.1095 mean=0.0018 std=0.9993


## 6. PyTorch Dataset & DataLoader


In [8]:
class SequenceDataset(Dataset):
    def __init__(self,x,y): self.x=torch.tensor(x); self.y=torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]
train_loader=DataLoader(SequenceDataset(Xtr,ytr),CONFIG["batch_size"],shuffle=True); val_loader=DataLoader(SequenceDataset(Xv,yv),CONFIG["batch_size"]); test_loader=DataLoader(SequenceDataset(Xte,yte),CONFIG["batch_size"])


## 7. Proposed Model Definition


In [9]:
class DiffusionGCN(nn.Module):

    def __init__(self, n, h):
        super().__init__()
        self.theta = nn.Linear(3, h)
        self.register_buffer('A', torch.eye(n))

    def set_graph(self, a):
        self.A.copy_(a)

    def forward(self, x):
        ax = torch.einsum('ij,btj->bti', self.A, x)
        a2x = torch.einsum('ij,btj->bti', self.A, ax)
        return self.theta(torch.stack([x, ax, a2x], -1))

class DiffusionTemporalTransformer(nn.Module):

    def __init__(self, n, h=64):
        super().__init__()
        self.gcn = DiffusionGCN(n, h)
        enc = nn.TransformerEncoderLayer(h, 4, 128, batch_first=True)
        self.temporal = nn.TransformerEncoder(enc, 2)
        self.head = nn.Linear(h, 1)

    def forward(self, x):
        g = self.gcn(x)
        b, t, n, h = g.shape
        # fold nodes into the batch dim so the temporal transformer keeps a
        # per-node representation instead of averaging nodes away first
        z = g.permute(0, 2, 1, 3).reshape(b * n, t, h)
        z = self.temporal(z)[:, -1]
        return self.head(z).reshape(b, n)
corr = np.corrcoef(series[:ntr], rowvar=False)
adj = np.nan_to_num(np.abs(corr))
np.fill_diagonal(adj, 1)
adj = (adj >= np.quantile(adj, 0.9)) * adj
adj = adj / (adj.sum(1, keepdims=True) + 1e-08)


## 8. Training Loop


In [10]:
from tqdm.auto import tqdm

def run_epoch(model, loader, criterion, optimizer=None, amp_scaler=None):
    model.train(optimizer is not None)
    total = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = (xb.to(DEVICE), yb.to(DEVICE))
        if optimizer:
            optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out = model(xb)
            loss = criterion(out, yb)
        if optimizer:
            if amp_scaler is not None:
                amp_scaler.scale(loss).backward()
                amp_scaler.step(optimizer)
                amp_scaler.update()
            else:
                loss.backward()
                optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / max(n, 1)

def train_model(model, train_loader, val_loader, checkpoint, classification=False):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss() if classification else nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    amp_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    history = {'train_loss': [], 'val_loss': []}
    best = float('inf')
    stale = 0
    for epoch in tqdm(range(CONFIG['epochs']), desc='Training', unit='epoch'):
        tr = run_epoch(model, train_loader, criterion, opt, amp_scaler)
        with torch.no_grad():
            va = run_epoch(model, val_loader, criterion)
        history['train_loss'].append(tr)
        history['val_loss'].append(va)
        scheduler.step(va)
        print(f'epoch={epoch + 1:02d} train={tr:.5f} val={va:.5f}')
        if va < best:
            best = va
            stale = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                break
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    return (model, history)

def predict(model, loader, classification=False):
    model.eval()
    pred = []
    true = []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(DEVICE)).cpu()
            pred.append(torch.softmax(out, 1) if classification else out)
            true.append(yb)
    return (torch.cat(pred).numpy(), torch.cat(true).numpy())

hybrid = DiffusionTemporalTransformer(series.shape[1])
hybrid.gcn.set_graph(torch.tensor(adj, dtype=torch.float32))
hybrid_ckpt = os.path.join(CONFIG['checkpoints_dir'], 'best_hybrid.pt')
hybrid, hybrid_history = train_model(hybrid, train_loader, val_loader, hybrid_ckpt)

/tmp/ipykernel_3154/1404822741.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Training:   0%|          | 0/30 [00:00<?, ?epoch/s]

epoch=01 train=0.11185 val=0.04365


epoch=02 train=0.04528 val=0.04333


epoch=03 train=0.04360 val=0.04301


...

epoch=14 train=0.04098 val=0.04155


epoch=15 train=0.04096 val=0.04084


epoch=16 train=0.04085 val=0.04066


epoch=17 train=0.04084 val=0.04074


epoch=18 train=0.04080 val=0.04090


## 9. Evaluation Metrics


In [ ]:
def eval_predictions(pred_scaled, y_scaled):
    pred = scaler.inverse_transform(pred_scaled.reshape(-1, series.shape[1])).reshape(pred_scaled.shape)
    y = scaler.inverse_transform(y_scaled.reshape(-1, series.shape[1])).reshape(y_scaled.shape)
    metrics = {'mae_speed_units': mean_absolute_error(y, pred), 'rmse_speed_units': mean_squared_error(y, pred) ** 0.5, 'r2': r2_score(y, pred, multioutput='variance_weighted')}
    return pred, y, metrics

results = {}
cached = {}
pred_scaled, y_scaled = predict(hybrid, test_loader)
pred, y, results['hybrid'] = eval_predictions(pred_scaled, y_scaled)
cached['hybrid'] = (pred, y)

# Persistence baseline: predict the last observed timestep for every sensor (test_loader
# is not shuffled, so this batch order lines up with y_scaled above).
xb_all = torch.cat([xb for xb, _ in test_loader]).numpy()
persist_scaled = xb_all[:, -1, :]
_, _, results['persistence_baseline'] = eval_predictions(persist_scaled, y_scaled)

# Reload-and-verify: load the checkpoint into a FRESH model instance and confirm the
# evaluation reproduces the in-memory result - catches save/load path bugs.
reloaded = DiffusionTemporalTransformer(series.shape[1])
reloaded.gcn.set_graph(torch.tensor(adj, dtype=torch.float32))
reloaded.load_state_dict(torch.load(hybrid_ckpt, map_location=DEVICE, weights_only=True))
reloaded = reloaded.to(DEVICE)
reload_pred_scaled, reload_y_scaled = predict(reloaded, test_loader)
_, _, reload_metrics = eval_predictions(reload_pred_scaled, reload_y_scaled)
assert abs(reload_metrics['mae_speed_units'] - results['hybrid']['mae_speed_units']) < 1e-3, f"Reloaded checkpoint mismatch: {reload_metrics} vs {results['hybrid']}"
print('Reload-and-verify OK:', reload_metrics)

Path(os.path.join(CONFIG['results_dir'], 'metrics.json')).write_text(json.dumps(results, indent=2), encoding='utf-8')
results

## 10. Required Figures


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(hybrid_history['train_loss'], label='hybrid train')
plt.plot(hybrid_history['val_loss'], label='hybrid val')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig01_loss_curves.png'), dpi=150)
plt.show()
if CONFIG['task_type'] == 'classification':
    probs, pred, y = cached['hybrid']
    cm = confusion_matrix(y, pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig02_confusion_matrix.png'), dpi=150)
    plt.show()
    support = np.bincount(y, minlength=probs.shape[1])
    per_class = [(pred[y == i] == i).mean() if (y == i).any() else np.nan for i in range(probs.shape[1])]
    plt.figure(figsize=(9, 4))
    plt.bar(range(len(per_class)), per_class)
    plt.ylabel('Per-class recall')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig03_class_performance.png'), dpi=150)
    plt.show()
else:
    pred, y = cached['hybrid']
    plt.figure(figsize=(6, 5))
    plt.scatter(y.ravel(), pred.ravel(), s=8, alpha=0.35)
    lo = min(y.min(), pred.min())
    hi = max(y.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], 'k--', label='y = x')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig02_predicted_vs_actual.png'), dpi=150)
    plt.show()
    residual = (pred - y).ravel()
    plt.figure(figsize=(7, 4))
    sns.histplot(residual, bins=40, kde=True)
    plt.xlabel('Residual')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig03_residual_distribution.png'), dpi=150)
    plt.show()
xb, yb = next(iter(test_loader))
xb = xb[:min(32, len(xb))].to(DEVICE).requires_grad_(True)
hybrid.zero_grad()
out = hybrid(xb)
score = out.max(1).values.sum() if out.ndim == 2 and out.shape[1] > 1 else out.sum()
score.backward()
importance = xb.grad.detach().abs().cpu().numpy()
imp = importance.mean(axis=tuple(range(importance.ndim - 1))) if importance.ndim > 2 else importance.mean(0)
plt.figure(figsize=(8, 4))
plt.plot(np.ravel(imp))
plt.title('Gradient-based input importance')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig04_feature_importance.png'), dpi=150)
plt.show()
if CONFIG['task_type'] == 'classification':
    errors = (cached['hybrid'][1] != cached['hybrid'][2]).astype(int)
else:
    errors = np.abs(cached['hybrid'][0] - cached['hybrid'][1]).reshape(len(cached['hybrid'][1]), -1).mean(1)
plt.figure(figsize=(8, 4))
plt.hist(errors, bins=30)
plt.title('Held-out error distribution')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig05_error_analysis.png'), dpi=150)
plt.show()
metric = next(iter(results['hybrid']))
plt.figure(figsize=(6, 4))
plt.bar(list(results.keys()), [results[k][metric] for k in results])
plt.ylabel(metric)
plt.title('Hybrid vs persistence baseline')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig06_proposed_metrics.png'), dpi=150)
plt.show()